In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
final_model = joblib.load("freight_rate_model.pkl")
final_kmeans = joblib.load("geographic_kmeans.pkl")

In [3]:
freight = pd.read_csv("train-test.csv")
december = pd.read_csv("december-chart-inputs.csv")

In [4]:
market_index_value = freight["market_index"].median()
quote_signal_value = freight["quote_signal"].median()

december["market_index"] = market_index_value
december["quote_signal"] = quote_signal_value

In [5]:
december.head(5)

,pickup,delivery,distance,equipment,weight,date,predicted_rate,market_index,quote_signal
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN,1.0558,2.05575
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN,1.0558,2.05575
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN,1.0558,2.05575
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN,1.0558,2.05575
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN,1.0558,2.05575


In [6]:
lexington_coords = freight.loc[
    freight["pickup"] == "Lexington",
    ["pickup_lat", "pickup_lon"]
].drop_duplicates()

print(lexington_coords)

    pickup_lat  pickup_lon
28    36.99152   -84.99876


In [7]:
fort_wayne_coords = freight.loc[
    freight["delivery"] == "Fort Wayne",
    ["delivery_lat", "delivery_lon"]
].drop_duplicates()

print(fort_wayne_coords)

    delivery_lat  delivery_lon
24      41.31561     -85.36206


In [8]:
december["pickup_lat"] = lexington_coords.iloc[0]["pickup_lat"]
december["pickup_lon"] = lexington_coords.iloc[0]["pickup_lon"]

december["delivery_lat"] = fort_wayne_coords.iloc[0]["delivery_lat"]
december["delivery_lon"] = fort_wayne_coords.iloc[0]["delivery_lon"]

In [9]:
december.head(5)


,pickup,delivery,distance,equipment,weight,date,predicted_rate,market_index,quote_signal,pickup_lat,pickup_lon,delivery_lat,delivery_lon
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN,1.0558,2.05575,36.99152,-84.99876,41.31561,-85.36206
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN,1.0558,2.05575,36.99152,-84.99876,41.31561,-85.36206
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN,1.0558,2.05575,36.99152,-84.99876,41.31561,-85.36206
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN,1.0558,2.05575,36.99152,-84.99876,41.31561,-85.36206
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN,1.0558,2.05575,36.99152,-84.99876,41.31561,-85.36206


In [10]:
december["date"] = pd.to_datetime(december["date"])

december["dayofweek"] = december["date"].dt.dayofweek
december["month"] = december["date"].dt.month

december["is_weekend"] = (
    december["dayofweek"].isin([5, 6]).astype(int)
)

december["is_month_end"] = (
    december["date"].dt.is_month_end.astype(int)
)

december["dow_sin"] = np.sin(
    2 * np.pi * december["dayofweek"] / 7
)

december["dow_cos"] = np.cos(
    2 * np.pi * december["dayofweek"] / 7
)

december["month_sin"] = np.sin(
    2 * np.pi * december["month"] / 12
)

december["month_cos"] = np.cos(
    2 * np.pi * december["month"] / 12
)

In [11]:
pickup = december[
    ["pickup_lat", "pickup_lon"]
].copy()

pickup.columns = ["lat", "lon"]

delivery = december[
    ["delivery_lat", "delivery_lon"]
].copy()

delivery.columns = ["lat", "lon"]

december["pickup_region"] = final_kmeans.predict(pickup)
december["delivery_region"] = final_kmeans.predict(delivery)

In [12]:
features = [
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "equipment",
    "pickup_region",
    "delivery_region",
    "is_weekend",
    "is_month_end",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

In [13]:
X_december = december[features]

december["predicted_rate"] = final_model.predict(
    X_december
).round(2)

In [14]:
december_predictions = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
].copy()

In [15]:
december_predictions.to_csv(
    "december_predictions.csv",
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.2f"
)